### Simple Gen-AI app with LangChain

### 1. Setting up the API's required

In [1]:
import os
from dotenv import load_dotenv,dotenv_values

load_dotenv() #We will load all the env variables
#Once we ran this, we do not need to run os.getenv() again, because we have already loaded the keys to the current process memory

values = dotenv_values(".env")  # returns a dict of key→value from that file, we check whether we have loaded the keys or not
print(values.keys()) #with this we can confirm, we have loaded the openai api key

odict_keys(['OPENAI_API_KEY', 'LANGCHAIN_API_KEY', 'LANGCHAIN_PROJECT', 'HF_TOKEN', 'LANGCHAIN_TRACING_V2'])


### 2. Data Ingestion

From a website we need to scrap the data

In [13]:
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader(web_paths=("https://docs.langchain.com/langsmith/observability-llm-tutorial",),)
#Here if we use web_paths , we can pass multiple of them as tuple, as we are passing only 1, we will put a comma
#Also there was an error saying set useragent ( like the browser agent etc ), we will just put comma and leave so that default browser agent will actually gets taken

docs = loader.load()
docs

[Document(metadata={'source': 'https://docs.langchain.com/langsmith/observability-llm-tutorial', 'title': 'Trace an LLM application tutorial - Docs by LangChain', 'language': 'en'}, page_content='Trace an LLM application tutorial - Docs by LangChainSkip to main contentDocs by LangChain home pageLangSmithSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationTrace an LLM application tutorialGet startedObservabilityEvaluationPrompt engineeringAgent deploymentPlatform setupReferenceOverviewQuickstartConceptsTrace an LLM applicationPolly AI assistantBetaTracing setupIntegrationsManual instrumentationThreadsConfiguration & troubleshootingProject & environment settingsCost trackingAdvanced tracing techniquesData & privacyTroubleshooting guidesViewing & managing tracesFilter tracesConfigure run previewsQuery traces (SDK)Compare tracesShare or unshare a trace publiclyRetrieve traces via CLILangSmith MCP ServerView server logs for a traceBulk export trace dataAutomationsSet up auto

Now, here we can see the entire content of the website we have scraped.

### 3. Data Transformation 

Text splitting in to chunks

In [14]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap=200)
doc_chunks=text_splitter.split_documents(docs)

doc_chunks

[Document(metadata={'source': 'https://docs.langchain.com/langsmith/observability-llm-tutorial', 'title': 'Trace an LLM application tutorial - Docs by LangChain', 'language': 'en'}, page_content='Trace an LLM application tutorial - Docs by LangChainSkip to main contentDocs by LangChain home pageLangSmithSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationTrace an LLM application tutorialGet startedObservabilityEvaluationPrompt engineeringAgent deploymentPlatform setupReferenceOverviewQuickstartConceptsTrace an LLM applicationPolly AI assistantBetaTracing setupIntegrationsManual instrumentationThreadsConfiguration & troubleshootingProject & environment settingsCost trackingAdvanced tracing techniquesData & privacyTroubleshooting guidesViewing & managing tracesFilter tracesConfigure run previewsQuery traces (SDK)Compare tracesShare or unshare a trace publiclyRetrieve traces via CLILangSmith MCP ServerView server logs for a traceBulk export trace dataAutomationsSet up auto

### 4. Vector Embeddings

Here we convert the chunks to vectors and then store them in a vectorDB to query for later

In [15]:
from langchain_community.embeddings import OllamaEmbeddings
from langchain_community.vectorstores import FAISS

#Please make sure that Ollama is active an running in the background
embeddings = OllamaEmbeddings(model ="qwen3-embedding:4b")

db = FAISS.from_documents(doc_chunks, embeddings)
db

#### 4.1 Similarity Search
Now, we can query the db with any question and retrieve the relevent chunks with the help of vector similarity

In [18]:
query = " It is also a good idea to start logging metadata. This allows you to start keep track of different attributes of your app."
result = db.similarity_search(query)

#Lets get the top most similar chunk and display its content
result[0].page_content

'You can also query for all runs with positive (or negative) feedback by using the filtering logic in the runs table. You can do this by creating a filter like the following:\n\n\u200bLogging metadata\nIt is also a good idea to start logging metadata. This allows you to start keep track of different attributes of your app. This is important in allowing you to know what version or variant of your app was used to produce a given result.\nFor this example, we will log the LLM used. Oftentimes you may be experimenting with different LLMs, so having that information as metadata can be useful for filtering. In order to do that, we can add it as such:\nCopyfrom openai import OpenAI\nfrom langsmith import traceable\nfrom langsmith.wrappers import wrap_openai\nopenai_client = wrap_openai(OpenAI())\n\n@traceable(run_type="retriever")\ndef retriever(query: str):\n    results = ["Harrison worked at Kensho"]\n    return results'

#### 4.2 Retrieval Chains & Document Chains

Above we have just taken a sentence a from the webpage and searched to check whether we get any relevent near by text to it with the help of vector search.

But, to ask any meaningful question instead of blindly copypasting the existing text, we need something called retrieval chains.

In [23]:
#This chains is not present in the latest version of langchain, we need to import it from langchain_classic
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

#Here we are giving our custom prompt
prompt=ChatPromptTemplate.from_template(
    """
    Answer the following question based only on the provided context:
    <context>
    {context}
    </context>
    """
)

Now, to create a document chain, we need an llm and a prompt

In [24]:
from langchain_ollama import ChatOllama
llm = ChatOllama(model='qwen3:4b')

Now that we have the llm, lets create a document chain

In [26]:
document_chain = create_stuff_documents_chain(llm,prompt)
document_chain

RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='\n    Answer the following question based only on the provided context:\n    <context>\n    {context}\n    </context>\n    '), additional_kwargs={})])
| ChatOllama(model='qwen3:4b')
| StrOutputParser(), kwargs={}, config={'run_name': 'stuff_documents_chain'}, config_factories=[])

This is nothing but instead of us creating a chain manually, it will create a chain for us with prompt_template --> llm --> output_parser

In [ ]:
from langchain_core.documents import Document
document_chain.invoke({
    "input": "Tell me about the logging metadata in langchain",
    "context": [Document(page_content=result[0].page_content)]
})

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


'Based solely on the provided context, the following answer addresses the implied question about logging metadata in LangSmith runs (as the specific question was not explicitly stated in the query):\n\n**Logging metadata in LangSmith runs is important to track different attributes of your app, which enables you to determine what version or variant of your app was used to produce a given result.** The context explicitly states this as a best practice, with an example of logging the LLM used (e.g., when experimenting with different models). It also notes that you can query runs with positive or negative feedback using filtering logic in the runs table. \n\nThis response is derived exclusively from the provided context and does not include external information.'

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


Here, we can use the chain by passing the input and the context dirctly to it, the context, we have taken from above ( which we got from the similarity search )

But, we do not want to do all this before similarity search work manually and then pass the context to this right.

--> We want to documents to come from the retriever that we will setup. That way we can use the retriever to dynamically select the most relevant documents and pass those in for a given question.

##### Retriver --> This is the key component which is needed for this automation, where it acts as an interface between my vectorDB and the query, so that we do not have to manually do the similarity search, it will do it for us

* Now how to create it?
Simply tell the database itself to act as a rertriever lol

In [30]:
retriever = db.as_retriever()

In [32]:
from langchain_classic.chains import create_retrieval_chain

retrieval_chain = create_retrieval_chain(retriever, document_chain) 
retrieval_chain


RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | VectorStoreRetriever(tags=['FAISS', 'OllamaEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x7b2c8d35ac90>, search_kwargs={}), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | ChatPromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='\n    Answer the following question based only on the provided context:\n    <context>\n    {context}\n    </context>\n    '), additional_kwargs={})])
  

Here, basically the context which the document-chain needed there manually, will be provided by the retriever and then combine them both to create a new chain called retriever_chain.

We can see above also, when we print, it , we can see additional information above regarding the getting of the similar embeddings from the documments automatically once we get the query.

#### 5. Final output

Here we just give it the question, and it answers , the context will be automatically fetched from the database that we have provided.


In [45]:
os.environ["LANGSMITH_DISABLE_RUN_COMPRESSION"] = "true"

In [ ]:
response = retrieval_chain.invoke(
    {"input": "how to log metadata in langsmith?"}
)

response["answer"]

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')
Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


'Based on the provided context, I can see that in the retriever function, the results are defined as `["Harrison worked at Kensho"]`. This appears in the code example where the retriever function is shown:\n\n```python\n@traceable(run_type="retriever")\ndef retriever(query: str):\n    results = ["Harrison worked at Kensho"]\n    return results\n```\n\nTherefore, according to the context, Harrison worked at Kensho.'

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


Now, we can see the reponse also contains the 'context' alongn with the 'answers'

In [48]:
response

{'input': 'how to log metadata in langsmith?',
 'context': [Document(id='5ec18f24-40e7-440c-a1d9-d1556d16432c', metadata={'source': 'https://docs.langchain.com/langsmith/observability-llm-tutorial', 'title': 'Trace an LLM application tutorial - Docs by LangChain', 'language': 'en'}, page_content='run_id = str(uuid7())\nrag(\n    "where did harrison work",\n    langsmith_extra={"run_id": run_id, "metadata": {"user_id": "harrison"}}\n)\n\nNow that we’ve logged these two pieces of metadata, we should be able to see them both show up in the UI here.\n\nWe can filter for these pieces of information by constructing a filter like the following:'),
  Document(id='61a36979-c3fc-4a1a-9dbd-460f1711f53c', metadata={'source': 'https://docs.langchain.com/langsmith/observability-llm-tutorial', 'title': 'Trace an LLM application tutorial - Docs by LangChain', 'language': 'en'}, page_content='You can also query for all runs with positive (or negative) feedback by using the filtering logic in the runs t

In [40]:
response['context']

[Document(id='5ec18f24-40e7-440c-a1d9-d1556d16432c', metadata={'source': 'https://docs.langchain.com/langsmith/observability-llm-tutorial', 'title': 'Trace an LLM application tutorial - Docs by LangChain', 'language': 'en'}, page_content='run_id = str(uuid7())\nrag(\n    "where did harrison work",\n    langsmith_extra={"run_id": run_id, "metadata": {"user_id": "harrison"}}\n)\n\nNow that we’ve logged these two pieces of metadata, we should be able to see them both show up in the UI here.\n\nWe can filter for these pieces of information by constructing a filter like the following:'),
 Document(id='61a36979-c3fc-4a1a-9dbd-460f1711f53c', metadata={'source': 'https://docs.langchain.com/langsmith/observability-llm-tutorial', 'title': 'Trace an LLM application tutorial - Docs by LangChain', 'language': 'en'}, page_content='You can also query for all runs with positive (or negative) feedback by using the filtering logic in the runs table. You can do this by creating a filter like the followin

In [41]:
response['answer']

'Based on the provided context, I can see that the answer to "where did harrison work" is Kensho.\n\nIn the context, there\'s a retriever function that returns: `["Harrison worked at Kensho"]`. This appears in the code snippet where the retriever function is defined with the `@traceable(run_type="retriever")` decorator.\n\nThe context shows that when the query "where did harrison work" is processed through this system, it returns the information that Harrison worked at Kensho.'